# Teaching the four ORCA event-count models

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sthsci/Orca/blob/main/notebooks/02_event_count_model_tutorial.ipynb)

**Teaching notebook.** Simulate the four population structures, inspect their observable signatures, and optionally recover the generating model with SMC Bayes factors.

Run the cells from top to bottom. Values collected near the start of each notebook are safe places to experiment. Bayesian SMC fitting is deliberately disabled by default in the analysis notebooks because it can take several minutes; set `RUN_INFERENCE = True` when the data checks and descriptive plots look right.

Use synthetic or approved anonymised data only. Do not upload names, clinical metadata, raw microscopy, or a donor key that could identify participants.


## 1. Model family

For cell $i$ observed for a common time $T$,

$$N_i\mid\lambda_i,T\sim\operatorname{Poisson}(\lambda_iT).$$

| Key | Population structure | Parameters |
|---|---|---|
| `homo` | one shared positive rate | $\lambda$ |
| `z2p` | structural nonengagers plus one shared positive rate | $\lambda,\phi_0$ |
| `dis2p` | Gamma-distributed positive rates | $\mu_\lambda,\sigma_\lambda$ |
| `hetero3` | structural nonengagers plus Gamma-distributed positive rates | $\mu_\lambda,\sigma_\lambda,\phi_0$ |

Zero inflation raises the fraction of zeros. Continuous rate heterogeneity usually creates overdispersion: the count variance exceeds the count mean. Those clues are useful, but model evidence evaluates the complete distributions.


In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys


def find_orca_checkout():
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src" / "bayesorca").is_dir():
            return candidate
    return None


ORCA_ROOT = find_orca_checkout()
if ORCA_ROOT is not None:
    sys.path[:0] = [str(ORCA_ROOT), str(ORCA_ROOT / "src")]
elif importlib.util.find_spec("bayesorca") is None:
    if sys.version_info[:2] != (3, 12):
        raise RuntimeError("ORCA currently requires a Python 3.12 Colab runtime.")
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "git+https://github.com/sthsci/Orca.git@main",
        ]
    )

import bayesorca

print("bayesorca", bayesorca.__version__)
print("Python", sys.version.split()[0])


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from bayesorca.event_counts import (
    MODEL_SPECS,
    InferenceSettings,
    evidence_table,
    run_count_models,
    simulate_event_counts,
    summary_table,
)

N_CELLS = 250
OBSERVATION_TIME = 1.0
MEAN_RATE = 4.0
SEED = 2026

MODEL_PARAMETERS = {
    "homo": {"sigma_lambda": 0.0, "p_zero": 0.0},
    "z2p": {"sigma_lambda": 0.0, "p_zero": 0.25},
    "dis2p": {"sigma_lambda": 2.0, "p_zero": 0.0},
    "hetero3": {"sigma_lambda": 2.0, "p_zero": 0.25},
}


In [ ]:
simulations = {}
truths = {}
for offset, (model_key, parameters) in enumerate(MODEL_PARAMETERS.items()):
    frame, truth = simulate_event_counts(
        model_key,
        n_cells=N_CELLS,
        obs_time=OBSERVATION_TIME,
        mu_lambda=MEAN_RATE,
        seed=SEED + offset,
        **parameters,
    )
    simulations[model_key] = frame
    truths[model_key] = truth

summary = pd.DataFrame(
    [
        {
            "model": MODEL_SPECS[key].short_label,
            "mean_count": frame["count"].mean(),
            "variance": frame["count"].var(),
            "variance/mean": frame["count"].var() / frame["count"].mean(),
            "zero_fraction": frame["count"].eq(0).mean(),
        }
        for key, frame in simulations.items()
    ]
)
summary.round(3)


In [ ]:
max_count = max(int(frame["count"].max()) for frame in simulations.values())
bins = np.arange(-0.5, max_count + 1.5)
fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True, sharey=True)

for ax, (model_key, frame) in zip(axes.flat, simulations.items()):
    ax.hist(frame["count"], bins=bins, color="#304B3D", alpha=0.82)
    ax.set_title(MODEL_SPECS[model_key].label)
    ax.set(xlabel="Events per cell", ylabel="Cells")

fig.suptitle("Observable counts under the four population structures")
fig.tight_layout()
plt.show()


## 2. Optional Bayesian recovery

The next cell fits every candidate model to the synthetic `hetero3` dataset. It is off by default so the notebook executes quickly. The 128-particle setting is for learning, not publication. Repeat runs and increase particles/chains before interpreting close Bayes factors.


In [ ]:
RUN_INFERENCE = False
DATASET_TO_FIT = "hetero3"

if RUN_INFERENCE:
    settings = InferenceSettings(draws=128, chains=1, cores=1, seed=2026)
    results = run_count_models(
        simulations[DATASET_TO_FIT],
        observation_time=OBSERVATION_TIME,
        settings=settings,
        model_keys=list(MODEL_SPECS),
    )
    display(evidence_table(results))
    display(summary_table(results))
else:
    print("Set RUN_INFERENCE = True to fit the four models.")


## Interpretation checklist

1. Confirm that the observation time has the intended units and is common to all rows.
2. Look at the empirical mean, variance, and zero fraction, but do not select a model from one statistic alone.
3. Rank models by marginal likelihood/Bayes factor and inspect posterior uncertainty under scientifically plausible models.
4. Treat a wide rate distribution as population heterogeneity, not proof of a particular molecular mechanism.
5. Record the generating seed here; for real analyses, record the input checksum and inference configuration instead.
